Applying Tversky transformations to environmental audio

In [1]:
# imports

import torch
from torch.utils.data import DataLoader
import torchvision.transforms as tt
import torchvision.models as models
import math

import torch.nn as nn

import torch.nn.functional as F

from pyha_analyzer.preprocessors import MelSpectrogramPreprocessors
from tqdm.notebook import tqdm

import numpy as np
import matplotlib.pyplot as plt

In [2]:
config = {
    "learning_rate": 2e-3,
    "learning_rate_decay": 0,
    "device": 'cuda',
    "seed": 1
}

In [3]:
torch.manual_seed(config["seed"])

In [4]:
from datasets import load_dataset

mnist_dataset = load_dataset("mnist")

train = mnist_dataset["train"]
test = mnist_dataset["test"]

def transform(batch):
  t = tt.Compose([
    tt.Grayscale(num_output_channels=3),
    tt.PILToTensor(),
    tt.ConvertImageDtype(torch.float)
    ])
  
  batch['image'] = [t(img) for img in batch['image']]
  batch['label'] = F.one_hot(torch.tensor(batch['label']), num_classes=10)
  
  return batch

train.set_transform(transform)
test.set_transform(transform)

In [5]:
transform

<function __main__.transform(batch)>

In [6]:
# # simplest implementation

# alpha = 0.1
# beta = 0.1
# theta = 0.1
# phi = torch.mul
# substract = True
# batch_size = (8,)
# feature_count = (20,)
# feature_shape = (1, 28, 28)
# class_count = (12,)

# batch = torch.rand(batch_size + feature_shape)

# f = torch.rand(feature_count + feature_shape)
# pi = torch.rand(class_count + feature_shape)


# output = []
# for a in batch:
#     batch_output = []
#     for p in pi:
#         a_f = torch.sum(a * f, (-1, -2, -3))
#         p_f = torch.sum(p * f, (-1, -2, -3))

#         val = phi(a_f, p_f)

#         mask = (torch.minimum(a_f, p_f) <= 0).int()

#         intersection = theta * (val * mask).sum()

#         if substract:
#             alpha_difference = -alpha * (a_f * ((a_f > 0) | (p_f <= 0)).int()).sum()
#             beta_difference = -beta * (p_f * ((p_f > 0) | (a_f <= 0)).int()).sum()
#         else:
#             alpha_difference = -alpha * ((a_f - p_f) * ((a_f > 0) | (p_f > 0) | (a_f > p_f)).int()).sum()
#             beta_difference = -beta * ((p_f - a_f) * ((p_f > 0) | (a_f > 0) | (p_f > a_f)).int()).sum()
        
#         batch_output.append(float(intersection + alpha_difference + beta_difference))
#     output.append(batch_output)
# output1 = torch.tensor(output)
# print(output1)

# def Tversky():
    
#     a_f = (batch.flatten(1, -1).unsqueeze(1) * f.flatten(1, -1).unsqueeze(0)).sum(-1).unsqueeze(1)
#     p_f = (pi.flatten(1, -1).unsqueeze(1) * f.flatten(1, -1).unsqueeze(0)).sum(-1).unsqueeze(0)
    

#     val = phi(a_f, p_f)
#     mask = (torch.minimum(a_f, p_f) <= 0).int()
#     intersection = theta * (val * mask).sum(-1)
    
#     if substract:
#         alpha_difference = -alpha * (a_f * ((a_f > 0) | (p_f <= 0)).int()).sum(-1)
#         beta_difference = -beta * (p_f * ((p_f > 0) | (a_f <= 0)).int()).sum(-1)
#     else:
#         alpha_difference = -alpha * ((a_f - p_f) * ((a_f > 0) | (p_f > 0) | (a_f > p_f)).int()).sum(-1)
#         beta_difference = -beta * ((p_f - a_f) * ((p_f > 0) | (a_f > 0) | (p_f > a_f)).int()).sum(-1)
    
#     return intersection + alpha_difference + beta_difference

In [7]:
class Tversky(nn.Module):
    """
    Similar to a fully-connected layer, but computes Tversky similarity instead
    """
    def __init__(
        self,
        in_features: tuple,
        out_features: int,
        num_features=256,
        feature_bank=None,
        phi=torch.mul,
        substract=True,
        device=None,
        dtype=None
    ) -> None:
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        
        self.phi = phi
        self.substract = substract
        
        if feature_bank is None:
            self.feature_bank = nn.Parameter(
                torch.empty((num_features,) + in_features, **factory_kwargs)
            )
        else:
            self.feature_bank = feature_bank
            
        self.prototypes = nn.Parameter(
                torch.empty( (out_features,) + in_features, **factory_kwargs)
            )
            
        self.alpha = nn.Parameter(torch.empty(1))
        self.beta = nn.Parameter(torch.empty(1))
        self.theta = nn.Parameter(torch.empty(1))

        self.reset_parameters()
        
    def reset_parameters(self) -> None:
        nn.init.uniform_(self.feature_bank)
        nn.init.uniform_(self.prototypes)
        nn.init.uniform_(self.alpha)
        nn.init.uniform_(self.beta)
        nn.init.uniform_(self.theta)
    
    def forward(self, input: torch.Tensor):
        
        a_f = (input.flatten(1, -1).unsqueeze(1) * self.feature_bank.flatten(1, -1).unsqueeze(0)).sum(-1).unsqueeze(1)
        p_f = (self.prototypes.flatten(1, -1).unsqueeze(1) * self.feature_bank.flatten(1, -1).unsqueeze(0)).sum(-1).unsqueeze(0)
        

        val = self.phi(a_f, p_f)
        mask = (torch.minimum(a_f, p_f) >= 0).int()
        intersection = self.theta * (val * mask).sum(-1)
        
        
        if self.substract:
            alpha_difference = -self.alpha * (a_f * ((a_f > 0) | (p_f <= 0)).int()).sum(-1)
            beta_difference = -self.beta * (p_f * ((p_f > 0) | (a_f <= 0)).int()).sum(-1)
        else:
            alpha_difference = -self.alpha * ((a_f - p_f) * ((a_f > 0) | (p_f > 0) | (a_f > p_f)).int()).sum(-1)
            beta_difference = -self.beta * ((p_f - a_f) * ((p_f > 0) | (a_f > 0) | (p_f > a_f)).int()).sum(-1)
        
        return intersection + alpha_difference + beta_difference
    
t = Tversky((4,), 10, 5)

A = torch.rand((2, 4))

t(A).shape

torch.Size([2, 10])

In [41]:
ResNet = models.resnet50(weights=None)
ResNet.fc = nn.Sequential(
    Tversky((ResNet.fc.in_features,),10, 20, substract=True)
)

# ResNet = models.resnet50(weights=None)
# ResNet.fc = nn.Linear(ResNet.fc.in_features, 10)

model = nn.Sequential(
    nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
    ResNet
).to(config['device'])

In [42]:
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0, lr=config["learning_rate"], betas=(0.8, 0.999))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda epoch : (1-config["learning_rate_decay"])**epoch)
metric = torch.nn.CrossEntropyLoss()
train_loader = DataLoader(mnist_dataset['train'], batch_size=8, num_workers=8, shuffle=True, pin_memory=True, pin_memory_device=config['device'])

In [ ]:
step, loss = 0, []

correct, total = 0, 0

for epoch in range(5):
    for data in tqdm(train_loader, desc=str(epoch), leave=False):
        img = data['image']

        optimizer.zero_grad()
        x = img.to(config["device"])
        
        pred = model(x).softmax(dim=-1)
        
        label = data['label'].to(config["device"], torch.float)
        
        score = metric(pred, label)
        score.backward()
        optimizer.step()
        
        loss.append(score.item())
        
        
        if step % 250 == 0:
            with torch.no_grad():
                print(np.mean(loss[-100:-1]))
                
                print(pred)
            
        step += 1

0:   0%|          | 0/7500 [00:00<?, ?it/s]

/home/a.jajodia.229/acoustic/anu_experiments_acoustic_species/.venv/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/a.jajodia.229/acoustic/anu_experiments_acoustic_species/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


nan
tensor([[7828793.5000, 7800515.5000, 7821860.5000, 7813026.5000, 7633877.0000,
         7559829.5000, 7623266.5000, 7639157.0000, 7633460.5000, 7921081.0000],
        [7431026.5000, 7404232.5000, 7424448.0000, 7416099.5000, 7246008.0000,
         7175800.5000, 7235938.0000, 7251056.5000, 7245598.5000, 7518668.5000],
        [6876135.5000, 6851319.5000, 6870114.0000, 6862297.0000, 6704962.5000,
         6639943.5000, 6695628.5000, 6709663.0000, 6704546.0000, 6957245.5000],
        [7720609.5000, 7692738.0000, 7713742.5000, 7705073.0000, 7528336.0000,
         7455409.5000, 7517862.0000, 7533585.5000, 7527963.5000, 7811611.0000],
        [8735817.0000, 8704229.0000, 8728047.0000, 8718239.0000, 8518258.0000,
         8435679.0000, 8506467.0000, 8524202.0000, 8517822.0000, 8838760.0000],
        [7979720.5000, 7950851.5000, 7972683.0000, 7963671.0000, 7780982.0000,
         7705568.5000, 7770203.0000, 7786512.0000, 7780590.5000, 8073717.5000],
        [7889052.0000, 7860580.5000, 78820

KeyboardInterrupt: 